In [13]:
!nvidia-smi

Thu Jul 16 04:59:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             33W /   70W |     459MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q langchain langchain_community langchain_classic langsmith \
    langchain_chroma langchain_groq langchain_huggingface \
    sentence-transformers pypdf gradio

In [ ]:
!pip install -q langchain langchain-groq langchain-chroma langchain-huggingface langchain-unstructured langsmith gradio "unstructured[html]"

In [ ]:
import os
import time
import requests

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA
from langchain_groq import ChatGroq
import gradio as gr

In [ ]:
from getpass import getpass

def get_secret(name):

    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass

    return getpass(f"Enter {name}: ")

os.environ["GROQ_API_KEY"] = get_secret("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = get_secret("LANGCHAIN_API_KEY")


os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "code-unnati-chatbot"

print("API keys and tracing configured successfully.")

In [ ]:
PDF_URL = "https://ai.stanford.edu/~nilsson/MLBOOK.pdf"
PDF_PATH = "ml.pdf"

response = requests.get(PDF_URL)
response.raise_for_status()

content_type = response.headers.get("Content-Type", "")
if "pdf" not in content_type.lower() and not response.content[:4] == b"%PDF":
    raise ValueError(
        f"The URL did not return a PDF (Content-Type: {content_type}). "
        "Update PDF_URL to point directly at the .pdf file."
    )

with open(PDF_PATH, "wb") as f:
    f.write(response.content)

print(f"Saved PDF ({len(response.content)} bytes) to {PDF_PATH}")

In [ ]:
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(documents)

assert len(texts) > 0, "No text extracted from the PDF — check the source file."
print(f"Loaded {len(documents)} pages, split into {len(texts)} chunks.")

In [ ]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
persist_directory = "vector_db"

chroma = Chroma.from_documents(
    documents=texts,
    embedding=embedding,
    persist_directory=persist_directory,
)

retriever = chroma.as_retriever(search_kwargs={"k": 4})
print("Vector store built and retriever ready.")


In [16]:
embedding=HuggingFaceEmbeddings()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
)

In [12]:
def get_response(question):
    if not question.strip():
        return "Please enter a question."
    try:
        start_time = time.time()
        response = qa_chain.invoke({"query": question})
        execution_time = time.time() - start_time
        print(f"Response generated in {execution_time:.2f} seconds")
        return response.get("result", "No response generated.")
    except Exception as e:
        return f"Error: {e}"


demo = gr.Interface(
    fn=get_response,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask a questions reated to pdf...",
        label="Your Question",
    ),
    outputs=gr.Textbox(lines=10, label="Response"),
    title=" Chatbot",
    description="Ask questions related to the CodeUnnati program.",
)

if __name__ == "__main__":
    demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8f70de6ee03a192495.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Response generated in 1.58 seconds
Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 

In [17]:
import os
os.system("jupyter nbconvert --to script notebook.ipynb")

65280

In [18]:
!jupyter nbconvert --to script your_notebook.ipynb

[NbConvertApp] WARNING | pattern 'your_notebook.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
 